# Qwen VQA + DINOv3 확대 영역 선택

Drive 마운트부터 제출 CSV 저장까지 위에서 아래로 실행하세요. **추론 전용이며 학습하지 않습니다.**
Qwen의 선지 TTA·선택적 Thinking·체크포인트는 유지합니다. 재검토 대상 중 문자/유사 선지 문항에만 DINOv3 크롭을 추가합니다.
DINOv3는 질문을 읽거나 OCR을 수행하지 않습니다. 패치 특징의 대비가 큰 영역을 고르는 휴리스틱이며 점수 향상은 검증이 필요합니다.
전체 이미지도 항상 함께 전달합니다. `facebook/dinov3-vitl16-pretrain-lvd1689m` 접근 승인을 받은 HF 계정으로 로그인해야 합니다. Google 계정은 달라도 됩니다.

기존 pipeline/AutoModel 예제를 실행했다면 런타임을 재시작한 뒤 이 파일만 위에서부터 실행하세요. 모델 중복 로딩을 피합니다.


In [ ]:
# 1. 실행 설정 — 이 셀을 바꾼 다음 아래 셀들을 순서대로 실행하세요.
INSTALL_PACKAGES = True
OFFLINE = False
MOUNT_DRIVE = True
MODEL_ID = "Qwen/Qwen3.8-27B"  # 완전 오프라인: 로컬 모델 폴더
MODEL_REVISION = "main"           # 재현 실험은 특정 commit으로 고정 가능
DATA_DIR_OVERRIDE = "/content/drive/MyDrive/ssafy-16-2-ai"
RUN_NAME = "qwen38_vqa_dinov3_v1"
RUN_MODE = "test"                 # "test" 또는 "validation"
PRESET = "thinking"               # baseline / binding / tta / crop / thinking
VALIDATION_PRESETS = ["baseline", "binding", "tta", "crop", "thinking"]
LABEL_COLUMN = "answer"           # train.csv의 정답 열
LABEL_MAP = {}                    # 숫자 라벨이면 명시: {"0":"a", "1":"b", ...}
VALIDATION_FRACTION = 0.2
VALIDATION_MAX_GROUPS = 128        # 0이면 선택된 검증 이미지 그룹 모두
SEED = 42
MAX_NEW_TOKENS = 32
THINK_TOKENS = 1536
EVIDENCE_TOKENS = 256              # Thinking 잘림/파싱 실패 시 비사고 근거 확인
REASONING_EFFORT = "low"
MARGIN_THRESHOLD = 0.15           # a-d 조건부 확률 차이. 보정된 정답 확률이 아님
FULL_MAX_PIXELS = 1536 * 1536
CROP_MAX_PIXELS = 1024 * 1024
OOM_PIXEL_SCALE = 0.5             # OOM 1회 축소 재시도, 축소 여부 기록
PIPELINE_VERSION = "v3.0-dinov3"
assert RUN_MODE in {"test", "validation"}
assert PRESET in {"baseline", "binding", "tta", "crop", "thinking"}
assert all(p in {"baseline", "binding", "tta", "crop", "thinking"} for p in VALIDATION_PRESETS)
assert 0 < VALIDATION_FRACTION < 1
assert not (OFFLINE and (INSTALL_PACKAGES or MOUNT_DRIVE)), "오프라인에서는 설치·Drive 마운트를 끄세요."

# DINOv3는 Qwen의 비전 인코더를 교체하지 않고 확대할 좌표만 선택합니다.
CROP_BACKEND = "dinov3"           # "fixed": 원본 노트북의 고정 크롭
DINO_MODEL_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"
DINO_REVISION = "main"            # 재현 실험에는 commit 사용
DINO_DEVICE = "cpu"              # Qwen VRAM 보존. 여유가 있으면 "cuda:0"
DINO_MAX_EDGE = 512               # DINO 분석 해상도. Qwen 크롭은 원본에서 추출
DINO_CROP_FRACTION = 0.60         # 가로·세로 각각 60%인 창 2개
DINO_MAX_IOU = 0.45               # 두 영역의 중복 제한
assert CROP_BACKEND in {"fixed", "dinov3"}
assert 0 < DINO_CROP_FRACTION < 1 and 0 <= DINO_MAX_IOU < 1
assert DINO_MAX_EDGE >= 16


In [ ]:
# 2. 패키지 설치 — 추론 서버/외부 LLM SDK는 설치하지 않습니다.
import sys, os, subprocess, importlib.metadata
if INSTALL_PACKAGES:
    before = {p: getattr(sys.modules[p], "__version__", None)
              for p in ("pandas", "transformers", "accelerate", "PIL") if p in sys.modules}
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "transformers>=5.8.0,<6", "accelerate", "pillow", "tqdm", "pandas==2.2.3"])
    for p, old in before.items():
        package = "pillow" if p == "PIL" else p
        if old != importlib.metadata.version(package):
            raise RuntimeError(f"{p} 버전 변경: 세션을 다시 시작하고 1번부터 실행하세요.")
if OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
from pathlib import Path
import json, re, hashlib, gc, time, math, types
from collections import Counter
from difflib import SequenceMatcher
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import torch
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration, set_seed
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
set_seed(SEED)

# 모델을 올리기 전에 승인 계정 인증을 준비합니다. 토큰을 파일/출력에 저장하지 않습니다.
if CROP_BACKEND == "dinov3" and not OFFLINE and not Path(DINO_MODEL_ID).is_dir():
    from huggingface_hub import get_token, login, hf_hub_download
    if not get_token():
        login()  # 승인받은 HF 계정의 Read 토큰을 입력창에 입력
    try:
        hf_hub_download(DINO_MODEL_ID, "config.json", revision=DINO_REVISION)
    except Exception as exc:
        raise RuntimeError("DINOv3 설정 파일 접근에 실패했습니다. 승인 계정의 토큰·읽기 권한·네트워크를 확인하세요. "
                           "Colab Secrets에 HF_TOKEN이 있으면 해당 계정도 확인하세요.") from exc
    print("DINOv3 설정 파일 접근 완료. 가중치는 모델 로딩 셀에서 읽습니다.")


In [ ]:
# 3. CPU 전용 도우미 — CSV 선지 문자열은 strip/lower 하지 않습니다.
LETTERS = "abcd"
BINDING = """질문의 대상과 요구 속성을 먼저 확인하세요. 표에서는 대상의 행과 열 제목을
함께 확인한 뒤 교차 셀을 읽으세요. 다른 대상의 숫자를 가져오지 마세요.
가장 많음/적음은 전체 대상의 개수를 비교하고, 큰 글씨나 큰 물체를 개수가 많은 것으로 해석하지 마세요.
아닌/없는 조건과 왼쪽/오른쪽/옆/뒤/위치의 기준 대상을 확인하세요.
문자, 숫자, 단위, 대소문자와 공백을 선지 그대로 비교하세요. 이미지에 없는 근거를 추측하지 마세요."""
CONDITION_RE = r"가장|많[은이]|적[은게]|아닌|아니|않|없는|제외|왼|오른|옆|뒤|앞|위에|아래|몇|개수|모두|합계|총|most|least|except|not |left|right|how many"
TEXT_RE = r"글|문자|적혀|쓰여|써[있진]|표기|이름|상호|브랜드|간판|가격|얼마|원|숫자|번호|시간|메뉴|SSID|와이파이|안타|득점|점수|name|text|price|number"

def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

def read_csv_exact(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False, na_filter=False)

def validate_frame(frame, labeled=False):
    required = {"id", "path", "question", *LETTERS}
    if labeled:
        required.add(LABEL_COLUMN)
    if not required.issubset(frame.columns):
        raise ValueError(f"CSV 누락 열: {required - set(frame.columns)}")
    if frame.empty or not frame["id"].is_unique or frame["id"].str.strip().eq("").any():
        raise ValueError("비어 있는 데이터/ID 또는 중복 ID가 있습니다.")
    for col in ("path", "question"):
        if frame[col].str.strip().eq("").any():
            raise ValueError(f"{col} 값이 비어 있습니다.")

def resolve_image(value, data_dir):
    value = str(value).replace("\\", "/")
    direct = Path(value)
    candidates = [direct] if direct.is_absolute() else []
    candidates.append(data_dir / value.removeprefix("/content/"))
    found = list(dict.fromkeys(p.resolve() for p in candidates if p.is_file()))
    if len(found) != 1:
        raise FileNotFoundError(f"이미지 경로를 하나로 확정할 수 없습니다: {value}; {found}")
    return found[0]

def option_order(shift):
    return tuple(LETTERS[(i + shift) % 4] for i in range(4))

def original_label(label, order):
    return order[LETTERS.index(label)] if label in list(LETTERS) else None

def parse_answer(raw, thinking=False):
    text = raw.strip()
    # 템플릿이 <think>를 미리 열었으므로 생성 문자열에 시작 태그가 없을 수 있습니다.
    if thinking:
        if "</think>" not in text:
            return None
        text = text.rsplit("</think>", 1)[1].strip()
    elif "<think>" in text or "</think>" in text:
        if "</think>" not in text:
            return None
        text = text.rsplit("</think>", 1)[1].strip()
    tags = re.findall(r"<answer>\s*([a-d])\s*</answer>", text, re.I)
    if tags:
        if len(tags) != 1 or not re.search(r"<answer>\s*[a-d]\s*</answer>\s*$", text, re.I):
            return None
        return tags[0].lower()
    # 문장 속 a, 선지 열거, 불완전 태그는 정답으로 간주하지 않습니다.
    return text.lower() if re.fullmatch(r"[a-d]", text, re.I) else None

def risk_flags(row):
    values = [row[c] for c in LETTERS]
    pairs = [(values[i], values[j]) for i in range(4) for j in range(i + 1, 4)]
    return {
        "condition": bool(re.search(CONDITION_RE, row["question"], re.I)),
        "text": bool(re.search(TEXT_RE, row["question"], re.I)),
        "similar_options": any(SequenceMatcher(None, a, b).ratio() >= .85 for a, b in pairs),
        "ambiguous_options": any(a.strip().casefold() == b.strip().casefold() for a, b in pairs),
    }

def bounded_image(image, max_pixels):
    if image.width * image.height <= max_pixels:
        return image.copy()
    scale = math.sqrt(max_pixels / (image.width * image.height))
    return image.resize((max(1, int(image.width * scale)), max(1, int(image.height * scale))), Image.Resampling.LANCZOS)


def fixed_crop_boxes(width, height):
    if width >= height:
        return [(0, 0, math.ceil(width * .6), height),
                (math.floor(width * .4), 0, width, height)]
    return [(0, 0, width, math.ceil(height * .6)),
            (0, math.floor(height * .4), width, height)]

def box_iou(a, b):
    inter = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    area = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1])
    return inter / max(1, area-inter)

def select_dino_boxes(scores, width, height):
    # 적분 영상으로 창별 평균을 계산합니다. 동점은 좌표 순으로 결정합니다.
    rows, cols = len(scores), len(scores[0])
    flat = [v for row in scores for v in row]
    if not all(math.isfinite(v) for v in flat):
        raise ValueError("DINO 특징 점수에 비정상 값이 있습니다.")
    if max(flat) - min(flat) < 1e-6:
        return fixed_crop_boxes(width, height), "flat_features_fixed_fallback"
    bh = max(1, math.ceil(rows * DINO_CROP_FRACTION))
    bw = max(1, math.ceil(cols * DINO_CROP_FRACTION))
    integral = [[0.0]*(cols+1) for _ in range(rows+1)]
    for y in range(rows):
        for x in range(cols):
            integral[y+1][x+1] = scores[y][x]+integral[y][x+1]+integral[y+1][x]-integral[y][x]
    candidates = []
    for y in range(rows-bh+1):
        for x in range(cols-bw+1):
            value = (integral[y+bh][x+bw]-integral[y][x+bw]-integral[y+bh][x]+integral[y][x])/(bh*bw)
            box = (math.floor(x*width/cols), math.floor(y*height/rows),
                   min(width, math.ceil((x+bw)*width/cols)), min(height, math.ceil((y+bh)*height/rows)))
            candidates.append((value, box))
    candidates.sort(key=lambda item: (-item[0], item[1]))
    selected = []
    for value, box in candidates:
        if all(box_iou(box, other) <= DINO_MAX_IOU for other in selected):
            selected.append(box)
            if len(selected) == 2:
                return selected, "feature_contrast"
    return fixed_crop_boxes(width, height), "insufficient_diversity_fixed_fallback"

@torch.inference_mode()
def dino_crop_boxes(image):
    # 한 이미지의 좌표만 캐시: 같은 문항의 TTA에서 DINO를 반복 실행하지 않습니다.
    global _dino_box_cache
    key = (image.size, hashlib.sha256(image.tobytes()).hexdigest())
    if _dino_box_cache is not None and _dino_box_cache[0] == key:
        return _dino_box_cache[1]
    patch = int(dino_model.config.patch_size)
    ratio = min(1.0, DINO_MAX_EDGE / max(image.size))
    size = tuple(max(patch, int(v*ratio)//patch*patch) for v in image.size)
    with image.resize(size, Image.Resampling.BICUBIC) as small:
        batch = dino_processor(images=small, do_resize=False, do_center_crop=False, return_tensors="pt")
    pixels = batch['pixel_values'].to(DINO_DEVICE, dtype=torch.float32)
    grid_h, grid_w = pixels.shape[-2]//patch, pixels.shape[-1]//patch
    output = dino_model(pixel_values=pixels)
    registers = int(dino_model.config.num_register_tokens)
    features = output.last_hidden_state[:, 1 + registers:, :].float()
    if features.shape[1] != grid_h*grid_w:
        raise RuntimeError("DINO 패치 좌표 수가 맞지 않습니다. DINOv3 ViT 모델의 패치/레지스터 설정을 확인하세요.")
    f = torch.nn.functional.normalize(features[0], dim=-1).reshape(grid_h, grid_w, -1)
    mean = torch.nn.functional.normalize(f.mean(dim=(0, 1)), dim=-1)
    global_contrast = (1 - (f*mean).sum(-1)).clamp_min(0)
    local = torch.zeros_like(global_contrast)
    count = torch.zeros_like(local)
    if grid_h > 1:
        delta = (1-(f[1:]*f[:-1]).sum(-1)).clamp_min(0)
        local[1:] += delta; local[:-1] += delta
        count[1:] += 1; count[:-1] += 1
    if grid_w > 1:
        delta = (1-(f[:,1:]*f[:,:-1]).sum(-1)).clamp_min(0)
        local[:,1:] += delta; local[:,:-1] += delta
        count[:,1:] += 1; count[:,:-1] += 1
    scores = (.7*global_contrast + .3*local/count.clamp_min(1)).cpu().tolist()
    boxes, method = select_dino_boxes(scores, *image.size)
    result = (boxes, {"backend": "dinov3", "method": method, "boxes": boxes,
                      "analysis_size": list(size), "patch_grid": [grid_h, grid_w], "register_tokens": registers})
    _dino_box_cache = (key, result)
    return result

def image_views(image, crops=False, scale=1.0):
    views = [bounded_image(image, max(1, int(FULL_MAX_PIXELS * scale)))]
    descriptions = ["이미지 1: 전체 원본 화면"]
    metadata = {"backend": "none", "boxes": [], "original_size": list(image.size)}
    try:
        if crops:
            if CROP_BACKEND == "dinov3":
                boxes, info = dino_crop_boxes(image)
                metadata.update(info)
            else:
                boxes = fixed_crop_boxes(*image.size)
                metadata.update(backend="fixed", boxes=boxes, method="fixed_strips")
            for i, box in enumerate(boxes, 2):
                with image.crop(box) as cropped:
                    views.append(bounded_image(cropped, max(1, int(CROP_MAX_PIXELS * scale))))
                descriptions.append(f"이미지 {i}: 원본 픽셀 좌표 {box}의 확대 영역")
        return views, "\n".join(descriptions), metadata
    except Exception:
        for view in views:
            view.close()
        raise

def make_prompt(row, order, style, view_description):
    if style == "baseline":
        return (f"{row['question']}\n" + "\n".join(f"({c}) {row[c]}" for c in LETTERS)
                + "\n\n이미지를 보고 정답을 고르세요. 설명 없이 a, b, c, d 중 소문자 한 글자만 출력하세요.")
    # JSON 따옴표로 끝 공백도 표현합니다. 값 자체를 정규화하지 않습니다.
    choices = "\n".join(f"({shown}) {json.dumps(row[original], ensure_ascii=False)}"
                        for shown, original in zip(LETTERS, order))
    ending = ("보이는 근거를 짧게 요약하고 마지막 줄에 <answer>a</answer> 형식으로 정답 하나를 출력하세요."
              if style == "evidence" else "설명 없이 a, b, c, d 중 소문자 한 글자만 출력하세요.")
    return (f"{view_description}\n확대 이미지들은 같은 장면의 일부입니다. 중복해서 세지 마세요. "
            "개수와 위치 관계는 이미지 1을 기준으로 판단하세요.\n"
            f"{BINDING}\n질문: {row['question']}\n선지(JSON 문자열):\n{choices}\n{ending}")

def choose_answer(votes, fallback):
    valid = [v["answer"] for v in votes if v.get("answer") in list(LETTERS)]
    counts = Counter(valid)
    if not counts:
        return None, "no_valid_answer"
    winner, count = counts.most_common(1)[0]
    if count >= 2 and count > len(valid) / 2:
        return winner, "majority"
    if fallback in list(LETTERS):
        return fallback, "keep_base_no_majority"
    return None, "unresolved_no_base"

def split_groups(frame):
    groups = sorted(frame["_image_sha"].unique(),
                    key=lambda g: hashlib.sha256(f"{SEED}:{g}".encode()).hexdigest())
    if len(groups) < 2:
        raise ValueError("이미지 그룹 검증에는 서로 다른 이미지가 2개 이상 필요합니다.")
    n = max(1, min(len(groups) - 1, round(len(groups) * VALIDATION_FRACTION)))
    if VALIDATION_MAX_GROUPS:
        n = min(n, VALIDATION_MAX_GROUPS)
    held_out = set(groups[:n])
    valid = frame[frame["_image_sha"].isin(held_out)].copy()
    rest = frame[~frame["_image_sha"].isin(held_out)].copy()
    assert set(valid["_image_sha"]).isdisjoint(rest["_image_sha"])
    return valid, rest


In [ ]:
# 4. 데이터 검증 및 이미지 해시 — 모델을 로딩하기 전에 파일 오류를 확인합니다.
root = Path("/content/drive/MyDrive")
candidates = ([Path(DATA_DIR_OVERRIDE)] if DATA_DIR_OVERRIDE else
              [root / "ai_2_challenge/data", root / "ssafy-16-2-ai", root / "data", root])
csv_name = "test.csv" if RUN_MODE == "test" else "train.csv"
DATA_DIR = next((p for p in candidates if (p / csv_name).is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError(f"{csv_name} 폴더를 DATA_DIR_OVERRIDE에 입력하세요.")
dataset = read_csv_exact(DATA_DIR / csv_name)
validate_frame(dataset, labeled=RUN_MODE == "validation")
paths, hashes = [], {}
for value in tqdm(dataset["path"], desc="이미지 파일 확인"):
    p = resolve_image(value, DATA_DIR)
    paths.append(str(p))
    if str(p) not in hashes:
        with Image.open(p) as im:
            im.verify()
        hashes[str(p)] = sha_file(p)
dataset["_resolved_path"] = paths
dataset["_image_sha"] = [hashes[p] for p in paths]
if RUN_MODE == "validation":
    dataset["_truth"] = dataset[LABEL_COLUMN].map(lambda s: LABEL_MAP.get(s, s)).str.strip().str.lower()
    if not dataset["_truth"].isin(list(LETTERS)).all():
        raise ValueError("정답은 단일 a-d여야 합니다. LABEL_COLUMN/LABEL_MAP을 확인하세요.")
    active_df, unused_train = split_groups(dataset)
    presets = VALIDATION_PRESETS
else:
    active_df = dataset.copy()
    presets = [PRESET]
    submission_template = read_csv_exact(DATA_DIR / "sample_submission.csv")
    assert {"id", "answer"}.issubset(submission_template.columns)
    assert submission_template["id"].is_unique
    assert set(submission_template["id"]) == set(dataset["id"])
print(f"{RUN_MODE}: {len(active_df)}문항, {active_df['_image_sha'].nunique()}이미지 그룹")


**실행 자원:** 원본 27B BF16 가중치만 약 54GB이며 이미지·캐시 메모리가 추가됩니다.
기존 노트북의 95GiB GPU와 같은 대용량 GPU를 대상으로 합니다. 아래 메모리 검사는 보수적 사전 검사이며
충분한 메모리를 보장하지 않습니다. OOM이면 입력 픽셀 수를 줄여 1회 재시도하고 로그에 남깁니다.
반복 OOM이면 설정의 픽셀/토큰 예산을 줄이고 새 RUN_NAME으로 실행하세요.

TTA는 기본 2회, 선택된 문항은 추가 2회(필요 시 동률 해소 1회) 추론합니다.
Thinking이 예산 안에 끝나지 않으면 미완성 사고를 답으로 쓰지 않고 비사고 근거 확인을 1회 수행합니다.
기본 greedy 추론은 비교의 재현성을 위한 실험 설정이며 최적 샘플링 설정이라는 의미는 아닙니다.


In [ ]:
# 5. 모델을 한 번만 로드합니다. API 키/서버/HTTP 추론 요청은 사용하지 않습니다.
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU 런타임에서 실행하세요.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("이 설정에는 BF16 지원 GPU가 필요합니다.")
load_key = (MODEL_ID, MODEL_REVISION, OFFLINE)
if globals().get("_loaded_model_key") not in (None, load_key):
    raise RuntimeError("다른 모델 설정이 이미 로드되어 있습니다. 세션을 다시 시작하세요.")
if globals().get("_loaded_model_key") != load_key:
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)}, 여유 {free / 2**30:.1f} GiB")
    if free / 2**30 < 65:
        raise RuntimeError("BF16 27B + 시각 입력을 위해 최소 65GiB 여유를 확보하세요.")
    common = dict(revision=MODEL_REVISION, local_files_only=OFFLINE, trust_remote_code=False)
    processor = AutoProcessor.from_pretrained(MODEL_ID, **common)
    model = Qwen3_5ForConditionalGeneration.from_pretrained(
        MODEL_ID, **common, dtype=torch.bfloat16,
        device_map={"": 0}, attn_implementation="sdpa").eval()
    _loaded_model_key = load_key
tokenizer = processor.tokenizer
# 특수 think 토큰도 반드시 보존한 채로 파싱합니다.
label_ids = [tokenizer.encode(c, add_special_tokens=False) for c in LETTERS]
if not all(len(ids) == 1 for ids in label_ids):
    raise RuntimeError("라벨 마진 계산은 a-d가 각각 단일 토큰인 토크나이저를 요구합니다.")
label_ids = [ids[0] for ids in label_ids]
probe = [{"role": "user", "content": [{"type": "text", "text": "test"}]}]
off = processor.apply_chat_template(probe, tokenize=False, add_generation_prompt=True, enable_thinking=False)
on = processor.apply_chat_template(probe, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=True, reasoning_effort=REASONING_EFFORT)
if off == on or not on.rstrip().endswith("<think>"):
    raise RuntimeError("모델 템플릿의 native Thinking 제어를 확인할 수 없습니다.")
print("모델과 Thinking 템플릿 확인 완료")


# 승인 계정으로 DINOv3 가중치를 다운로드합니다. 추론은 로컬에서 수행합니다.
from transformers import AutoImageProcessor, AutoModel
_dino_box_cache = None
dino_manifest = {"backend": CROP_BACKEND}
if CROP_BACKEND == "dinov3":
    dino_key = (DINO_MODEL_ID, DINO_REVISION, DINO_DEVICE, OFFLINE)
    if globals().get("_loaded_dino_key") not in (None, dino_key):
        raise RuntimeError("DINO 설정이 바뀌었습니다. 런타임을 재시작하세요.")
    if globals().get("_loaded_dino_key") is None:
        dino_processor = AutoImageProcessor.from_pretrained(DINO_MODEL_ID, revision=DINO_REVISION,
            local_files_only=OFFLINE, trust_remote_code=False)
        dino_model = AutoModel.from_pretrained(DINO_MODEL_ID, revision=DINO_REVISION,
            local_files_only=OFFLINE, trust_remote_code=False, dtype=torch.float32).to(DINO_DEVICE).eval()
        _loaded_dino_key = dino_key
    dino_path = Path(DINO_MODEL_ID)
    dino_manifest.update(model=DINO_MODEL_ID, revision=DINO_REVISION,
        resolved_revision=getattr(dino_model.config, "_commit_hash", None),
        model_config=dino_model.config.to_dict(), processor=dino_processor.to_dict(),
        local_files=([[str(p.relative_to(dino_path)), p.stat().st_size, p.stat().st_mtime_ns]
                      for p in sorted(dino_path.rglob("*")) if p.is_file()] if dino_path.is_dir() else None),
        device=DINO_DEVICE, max_edge=DINO_MAX_EDGE, fraction=DINO_CROP_FRACTION, max_iou=DINO_MAX_IOU)
    print(f"DINOv3 준비: {DINO_MODEL_ID} / {DINO_DEVICE}; 재검토 크롭에서만 특징 추출")


In [ ]:
# 6. 실제 GPU 추론. 근거는 짧은 요약으로 요청하고 원문은 진단 로그로만 저장합니다.
@torch.inference_mode()
def generate_vote(row, image, shift=0, style="binding", crops=False, thinking=False):
    order = option_order(shift)
    for attempt in range(2):
        views = inputs = outputs = None
        crop_metadata = {"backend": "none", "boxes": []}
        try:
            scale = 1.0 if attempt == 0 else OOM_PIXEL_SCALE
            # baseline은 기존의 전체 해상도/프롬프트를 그대로 재현합니다.
            if style == "baseline" and attempt == 0:
                views, description = [image.copy()], "이미지 1: 전체 원본 화면"
            else:
                views, description, crop_metadata = image_views(image, crops, scale)
            prompt = make_prompt(row, order, style, description)
            messages = [{"role": "user", "content":
                        [{"type": "image", "image": im} for im in views]
                        + [{"type": "text", "text": prompt}]}]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=thinking, reasoning_effort=REASONING_EFFORT)
            inputs = processor(text=[text], images=views, return_tensors="pt").to(model.device)
            budget = THINK_TOKENS if thinking else (EVIDENCE_TOKENS if style == "evidence" else MAX_NEW_TOKENS)
            outputs = model.generate(**inputs, max_new_tokens=budget, do_sample=False,
                num_beams=1, repetition_penalty=1.0,
                return_dict_in_generate=True, output_scores=(style != "evidence"))
            ids = outputs.sequences[0, inputs["input_ids"].shape[-1]:]
            raw = tokenizer.decode(ids, skip_special_tokens=False)
            # 종료 제어 토큰만 제거하고 <think>와 </think>는 보존합니다.
            clean = raw
            for marker in ("<|im_end|>", "<|endoftext|>", "<|im_start|>assistant"):
                clean = clean.replace(marker, "")
            displayed = parse_answer(clean, thinking)
            margin, probabilities = None, None
            if style != "evidence" and len(ids) and int(ids[0]) in label_ids and displayed:
                p = outputs.scores[0][0, label_ids].float().softmax(-1).cpu().tolist()
                ranked = sorted(p, reverse=True)
                margin = ranked[0] - ranked[1]
                probabilities = {original: p[i] for i, original in enumerate(order)}
            return dict(answer=original_label(displayed, order), raw=raw,
                shift=shift, order=list(order), style=style, crops=crops, thinking=thinking,
                margin=margin, probabilities=probabilities, tokens=len(ids),
                hit_token_budget=len(ids) >= budget, pixel_scale=scale,
                parse_ok=displayed is not None, crop_metadata=crop_metadata)
        except torch.cuda.OutOfMemoryError:
            if attempt == 1:
                raise
        finally:
            del inputs, outputs
            if views:
                for view in views:
                    view.close()
            gc.collect()
            torch.cuda.empty_cache()

def infer_row(row, preset):
    # 정답 열을 받지 않으며 리뷰에 기존 예측을 알려주지 않습니다.
    flags = risk_flags(row)
    with Image.open(row["_resolved_path"]) as src:
        image = ImageOps.exif_transpose(src).convert("RGB")
    votes = []
    try:
        base = generate_vote(row, image, style="baseline" if preset == "baseline" else "binding")
        votes.append(base)
        fallback = base["answer"]
        if preset in {"baseline", "binding"}:
            return dict(answer=fallback, base_answer=fallback, votes=votes, flags=flags,
                        reviewed=False, decision="single_pass", risk_reasons=[])
        second = generate_vote(row, image, shift=1)
        votes.append(second)
        reasons = []
        if base["answer"] != second["answer"]:
            reasons.append("tta_disagreement")
        if any(v["answer"] is None for v in votes):
            reasons.append("parse_failure")
        if any(v["margin"] is None or v["margin"] < MARGIN_THRESHOLD for v in votes):
            reasons.append("low_or_unknown_margin")
        reasons += [key for key in ("condition", "similar_options", "ambiguous_options") if flags[key]]
        # 모든 OCR 질문을 재추론하지 않고 낮은 마진·조건 위험과 함께 선별합니다.
        reviewed = bool(reasons) and preset in {"crop", "thinking"}
        if reviewed:
            use_crops = flags["text"] or flags["similar_options"]
            def review(shift):
                if preset == "crop":
                    vote = generate_vote(row, image, shift=shift, crops=use_crops)
                else:
                    vote = generate_vote(row, image, shift=shift, style="evidence",
                                         crops=use_crops, thinking=True)
                votes.append(vote)
                if vote["answer"] is None and preset == "thinking":
                    retry = generate_vote(row, image, shift=shift, style="evidence",
                                          crops=use_crops, thinking=False)
                    retry["recovery"] = "thinking_parse_failure"
                    votes.append(retry)
            review(2)
            review(3)
            # 두 기본 답과 두 리뷰가 서로 갈리면 전체 화면만으로 독립 판정 1회.
            valid = [v["answer"] for v in votes if v["answer"]]
            counts = Counter(valid)
            if len(counts) > 1 and max(counts.values()) * 2 == len(valid):
                votes.append(generate_vote(row, image, shift=0, style="evidence", crops=False,
                                           thinking=(preset == "thinking")))
        answer, decision = choose_answer(votes, fallback)
        return dict(answer=answer, base_answer=fallback, votes=votes, flags=flags,
                    reviewed=reviewed, decision=decision, risk_reasons=reasons)
    finally:
        image.close()


**집계의 의미:** 마진은 첫 출력이 라벨 토큰일 때만 a–d 사이의 상대 점수 차이로 계산합니다.
그 외에는 unknown으로 처리합니다. 같은 모델의 표는 서로 독립된 확률 표본이 아니며, 다수결은 정답 보장이 아닙니다.
`crop` 단계는 위험 문항의 판독 재확인, `thinking` 단계는 같은 경로에 native Thinking을 추가한 실험입니다.
고신뢰 오답이나 키워드로 잡히지 않는 오류는 재검토에서 빠질 수 있습니다.

`baseline`은 기존 프롬프트·전체 해상도·32토큰·비사고 추론을 비교용으로 유지합니다.
다만 EXIF 방향 보정, 엄격한 파서는 적용되며, 기존 제출과 완전히 동일한 결과를 보장하지 않습니다.
`binding`은 프롬프트/문자열 표현과 입력 픽셀 제한을 함께 변경하므로 순수 프롬프트만의 기여도는 아닙니다.


In [ ]:
# 7. 체크포인트: 설정/코드/모델/CSV/이미지가 달라지면 기존 결과와 혼합하지 않습니다.
def atomic_json(path, data):
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)

def constant_signature(value):
    if isinstance(value, types.CodeType):
        return code_signature(value)
    if isinstance(value, frozenset):
        return ["frozenset", sorted((constant_signature(x) for x in value), key=repr)]
    if isinstance(value, tuple):
        return ["tuple", [constant_signature(x) for x in value]]
    return repr(value)

def code_signature(code):
    return [code.co_code.hex(), list(code.co_names),
            [constant_signature(x) for x in code.co_consts]]

function_names = ["parse_answer", "option_order", "original_label", "make_prompt", "risk_flags",
                  "bounded_image", "image_views", "choose_answer", "infer_row", "generate_vote", "split_groups"]
function_names += ["fixed_crop_boxes", "box_iou", "select_dino_boxes", "dino_crop_boxes"]
engine = {}
for name in function_names:
    fn = globals()[name]
    while hasattr(fn, "__wrapped__"):
        fn = fn.__wrapped__
    engine[name] = code_signature(fn.__code__)
engine_hash = hashlib.sha256(json.dumps(engine, sort_keys=True).encode()).hexdigest()
model_path = Path(MODEL_ID)
local_model_files = None
if model_path.is_dir():
    local_model_files = [[str(p.relative_to(model_path)), p.stat().st_size, p.stat().st_mtime_ns]
                         for p in sorted(model_path.rglob("*")) if p.is_file()]
common_manifest = dict(
    pipeline=PIPELINE_VERSION, engine=engine_hash, dino=dino_manifest, model=MODEL_ID, revision=MODEL_REVISION,
    resolved_revision=getattr(model.config, "_commit_hash", None), local_model_files=local_model_files,
    model_config=model.config.to_dict(),
    chat_template=processor.chat_template, mode=RUN_MODE,
    csv_sha=sha_file(DATA_DIR / csv_name),
    images=sorted(set(active_df["_image_sha"])), ids=active_df["id"].tolist(),
    seed=SEED, max_tokens=MAX_NEW_TOKENS, think_tokens=THINK_TOKENS, evidence_tokens=EVIDENCE_TOKENS,
    reasoning_effort=REASONING_EFFORT, margin=MARGIN_THRESHOLD,
    full_pixels=FULL_MAX_PIXELS, crop_pixels=CROP_MAX_PIXELS, oom_scale=OOM_PIXEL_SCALE,
    prompt=BINDING, condition_re=CONDITION_RE, text_re=TEXT_RE,
    label_column=LABEL_COLUMN, label_map=LABEL_MAP,
    versions={p: importlib.metadata.version(p) for p in ("torch", "transformers", "pandas", "pillow", "accelerate")})
# JSON roundtrip으로 tuple/int key 등의 표현을 정규화합니다.
common_manifest = json.loads(json.dumps(common_manifest, default=str))
OUT_DIR = DATA_DIR / "qwen_results" / RUN_NAME / RUN_MODE
OUT_DIR.mkdir(parents=True, exist_ok=True)
if RUN_MODE == "validation":
    active_df[["id", "path", "_image_sha"]].to_csv(OUT_DIR / "validation_split.csv", index=False)
    unused_train[["id", "path", "_image_sha"]].to_csv(OUT_DIR / "unused_train_split.csv", index=False)

def run_preset(preset):
    folder = OUT_DIR / preset
    folder.mkdir(exist_ok=True)
    cache_dir = folder / "items"
    cache_dir.mkdir(exist_ok=True)
    manifest_path = folder / "run_config.json"
    manifest = dict(common_manifest, preset=preset)
    if manifest_path.exists():
        if json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
            raise RuntimeError(f"{folder}: 기존 실행과 설정이 다릅니다. 새 RUN_NAME을 사용하세요.")
    else:
        if any(cache_dir.iterdir()):
            raise RuntimeError("설정 기록 없는 체크포인트입니다. 새 RUN_NAME을 사용하세요.")
        atomic_json(manifest_path, manifest)
    records = []
    for _, full_row in tqdm(active_df.iterrows(), total=len(active_df), desc=preset):
        # 모델에 전달할 열을 화이트리스트로 제한: 정답/분석 문서/test ID 힌트 미사용.
        row = {c: full_row[c] for c in ("id", "question", "a", "b", "c", "d", "_resolved_path")}
        item = cache_dir / (hashlib.sha256(row["id"].encode()).hexdigest() + ".json")
        if item.exists():
            result = json.loads(item.read_text(encoding="utf-8"))
            if result.get("id") != row["id"] or result.get("answer") not in [None, *LETTERS]:
                raise ValueError(f"손상된 체크포인트: {item}")
        else:
            start = time.perf_counter()
            try:
                result = infer_row(row, preset)
            except Exception as exc:
                atomic_json(folder / "last_error.json", {"id": row["id"], "error": repr(exc)})
                raise  # 미해결 시스템 오류를 임의의 답으로 감추지 않습니다.
            result.update(id=row["id"], elapsed_seconds=time.perf_counter() - start)
            atomic_json(item, result)  # 각 문항마다 원자적 저장: 중단 후 재개 가능
        records.append(result)
    summary = pd.DataFrame([dict(id=r["id"], answer=r["answer"], base_answer=r["base_answer"],
        reviewed=r["reviewed"], decision=r["decision"], changed=r["answer"] != r["base_answer"],
        calls=len(r["votes"]), tokens=sum(v["tokens"] for v in r["votes"]),
        parse_failures=sum(not v["parse_ok"] for v in r["votes"]),
        truncated_calls=sum(v["hit_token_budget"] for v in r["votes"]),
        oom_retries=sum(v["pixel_scale"] != 1 for v in r["votes"]),
        dino_calls=sum(v.get("crop_metadata", {}).get("backend") == "dinov3" for v in r["votes"]),
        seconds=r["elapsed_seconds"], risk_reasons="|".join(r["risk_reasons"]), **r["flags"])
        for r in records])
    summary.to_csv(folder / "diagnostics.csv", index=False)
    if RUN_MODE == "test":
        if summary["answer"].isna().any():
            raise RuntimeError(f"해결되지 않은 답이 있습니다. {folder / 'diagnostics.csv'}를 확인하세요. 제출 파일을 만들지 않았습니다.")
        submission = submission_template.copy()
        submission["answer"] = submission["id"].map(summary.set_index("id")["answer"])
        assert submission["answer"].isin(list(LETTERS)).all()
        assert submission["id"].tolist() == submission_template["id"].tolist()
        temp = folder / "submission.tmp.csv"
        submission.to_csv(temp, index=False)
        temp.replace(folder / "submission.csv")
        print(f"제출 파일: {folder / 'submission.csv'}")
    return summary


In [ ]:
# 8. 실행 — 검증 모드에서는 5개 설정을 같은 이미지 그룹에 적용합니다.
required_state = {
    "RUN_MODE": "1번 실행 설정", "PRESET": "1번 실행 설정",
    "VALIDATION_PRESETS": "1번 실행 설정", "active_df": "4번 데이터 준비",
    "model": "5번 모델 로딩", "processor": "5번 모델 로딩",
    "infer_row": "6번 추론 함수", "run_preset": "7번 체크포인트",
    "common_manifest": "7번 체크포인트", "OUT_DIR": "7번 체크포인트",
}
missing_state = [f"{name} ({step})" for name, step in required_state.items() if name not in globals()]
if missing_state:
    raise RuntimeError("실행 준비가 완료되지 않았습니다. 다음 셀을 위에서부터 실행하세요: "
                       + ", ".join(missing_state))
if RUN_MODE not in {"test", "validation"}:
    raise ValueError("RUN_MODE는 test 또는 validation이어야 합니다.")
if common_manifest.get("mode") != RUN_MODE:
    raise RuntimeError("RUN_MODE가 데이터 준비 이후 변경되었습니다. 4번부터 7번까지 다시 실행하세요.")
if RUN_MODE == "test" and "submission_template" not in globals():
    raise RuntimeError("제출 양식이 준비되지 않았습니다. 4번 데이터 준비 셀을 실행하세요.")
if RUN_MODE == "validation" and "_truth" not in active_df.columns:
    raise RuntimeError("검증 정답이 준비되지 않았습니다. 4번 데이터 준비 셀을 실행하세요.")
# 4번 셀의 임시 변수에 의존하지 않고 현재 실행 설정에서 항상 계산합니다.
presets = list(VALIDATION_PRESETS) if RUN_MODE == "validation" else [PRESET]
if not presets or any(p not in {"baseline", "binding", "tta", "crop", "thinking"} for p in presets):
    raise ValueError("PRESET 또는 VALIDATION_PRESETS를 확인하세요.")
results = {preset: run_preset(preset) for preset in presets}
if RUN_MODE == "validation":
    truth = active_df.set_index("id")["_truth"]
    baseline = results.get("baseline")
    baseline_correct = (baseline.set_index("id")["answer"].eq(truth) if baseline is not None else None)
    metrics = []
    for preset, frame in results.items():
        table = frame.set_index("id").reindex(truth.index).copy()
        table["truth"] = truth
        table["correct"] = table["answer"].eq(truth)
        m = dict(preset=preset, n=len(table), accuracy=float(table["correct"].mean()),
                 unresolved=int(table["answer"].isna().sum()),
                 review_rate=float(table["reviewed"].mean()),
                 mean_calls=float(table["calls"].mean()),
                 mean_tokens=float(table["tokens"].mean()),
                 mean_seconds=float(table["seconds"].mean()))
        if baseline_correct is not None:
            b = baseline_correct.reindex(truth.index)
            m.update(fixed=int((~b & table["correct"]).sum()),
                     broken=int((b & ~table["correct"]).sum()),
                     net_gain=int((~b & table["correct"]).sum() - (b & ~table["correct"]).sum()))
        table.to_csv(OUT_DIR / preset / "validation_details.csv")
        groups = table.groupby(["text", "condition", "similar_options"], dropna=False)["correct"].agg(["mean", "count"])
        groups.to_csv(OUT_DIR / preset / "risk_group_accuracy.csv")
        metrics.append(m)
    comparison = pd.DataFrame(metrics)
    comparison.to_csv(OUT_DIR / "validation_comparison.csv", index=False)
    display(comparison)
    print("fixed(오답 교정)와 broken(기존 정답 훼손)을 함께 비교하세요. 작은 검증셋의 차이는 확정적 개선 근거가 아닙니다.")
else:
    display(results[PRESET].head())


## 결과 확인 및 다음 실험

- `.../qwen_results/<RUN_NAME>/test/<PRESET>/submission.csv`: 제출 양식의 순서를 보존한 최종 답안.
- `diagnostics.csv`: 기본 답과 최종 답, 변경 여부, 위험 태그, 호출 수, 파싱 실패, 잘림, OOM 기록.
- `items/*.json`: 각 문항의 선지 역매핑, 모든 추론 출력과 마진, 최종 선택 근거. 정답 라벨은 모델 입력에 포함하지 않습니다.
- 검증 모드의 `validation_comparison.csv`: 기준선 대비 교정·훼손·순증가와 비용 비교.

검증에서 개선을 확인한 설정을 `PRESET`에 지정하고 `RUN_MODE="test"`로 실행하세요.
여러 설정을 비교하며 선택한 검증셋은 튜닝용입니다. 최종 성능 추정에는 별도의 미사용 검증셋이 필요합니다.
이미지 그룹은 파일 내용 SHA256 기준이므로 재인코딩·약간 수정된 유사 이미지까지 자동 묶지는 않습니다.
점수가 오른다는 실측 결과는 아직 없습니다. 첨부 파일에는 실제 데이터셋/정답이 없고 수정 환경에 GPU도 없어
실제 모델 추론과 제출 점수 검증은 수행하지 않았습니다.

참고한 공식 자료: [Qwen 모델 카드](https://huggingface.co/Qwen/Qwen3.8-27B),
[Qwen Thinking 템플릿](https://huggingface.co/Qwen/Qwen3.8-27B/blob/main/chat_template.jinja),
[Transformers 모델 문서](https://huggingface.co/docs/transformers/main/en/model_doc/qwen3_5).
공식 자료의 API 예제는 사용하지 않고, 로컬 Transformers 추론과 템플릿 설정만 반영했습니다.


### DINOv3 실험
- 기본 설정은 테스트 전체를 추론하고 `qwen_results/qwen38_vqa_dinov3_v1/test/thinking/submission.csv`를 저장합니다.
- 각 `items/*.json`의 `votes[].crop_metadata`에 원본 좌표, 패치 격자, 고정 크롭 대체 여부가 기록됩니다.
- `diagnostics.csv`의 `dino_calls`는 DINO 좌표를 사용한 Qwen 호출 수입니다. DINO 특징 추출 횟수가 아닙니다.
- 성능 비교는 `RUN_MODE="validation"`, `VALIDATION_PRESETS=["thinking"]`으로 수행하세요. 동일 SEED에서
  `CROP_BACKEND="fixed"`, 새 `RUN_NAME`으로 다시 실행하면 같은 검증 문항에 대한 고정 크롭 결과가 나옵니다.
- DINOv3의 특징 대비는 질문과 무관합니다. 작은 글씨·표·개수 질문에서 고정 크롭보다 나빠질 수도 있습니다.
- 모델 가중치는 다운로드하지만 추론 API는 쓰지 않습니다. 완전 오프라인 실행은 두 모델을 미리 저장해야 합니다.
- CPU의 DINO 처리 속도가 느리면 GPU 메모리를 확인한 뒤 `DINO_DEVICE="cuda:0"`으로 바꾸고 재시작하세요.

공식 패치/레지스터 처리: https://huggingface.co/docs/transformers/model_doc/dinov3
첫 실행에서 인증 창이 나오면 승인 계정의 토큰으로 로그인하세요. 모델 다운로드 성공 전에는 접근 성공을 단정하지 않습니다.
